In [ ]:
import os
import time
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
import cv2
import yaml
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from torchvision.ops import RoIAlign
from PIL import Image

# Configuração do dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Usando dispositivo: {device}")

# Carregar configurações do dataset
print("📂 Carregando configurações do dataset...")
with open("dataset.yaml", "r") as file:
    dataset_config = yaml.safe_load(file)

train_dir = dataset_config["train"]
val_dir = dataset_config["val"]
nc = dataset_config["nc"]
class_names = dataset_config["names"]
print("✅ Configurações carregadas com sucesso!")

# Criando o Backbone do Zero
class CustomBackbone(nn.Module):
    def __init__(self):
        super(CustomBackbone, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(256)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        return x

# Criando a RPN (Region Proposal Network)
class RPN(nn.Module):
    def __init__(self, in_channels):
        super(RPN, self).__init__()
        self.conv = nn.Conv2d(in_channels, 512, kernel_size=3, padding=1)
        self.cls_logits = nn.Conv2d(512, 9 * 2, kernel_size=1)
        self.bbox_pred = nn.Conv2d(512, 9 * 4, kernel_size=1)

    def forward(self, x):
        x = F.relu(self.conv(x))
        logits = self.cls_logits(x)
        bbox_reg = self.bbox_pred(x)
        return logits, bbox_reg

# Criando ROI Pooling
class ROIPooling(nn.Module):
    def __init__(self, output_size):
        super(ROIPooling, self).__init__()
        self.roi_align = RoIAlign(output_size, spatial_scale=1.0, sampling_ratio=2)

    def forward(self, x, proposals):
        return self.roi_align(x, proposals)

# Criando a Cabeça de Classificação
class DetectionHead(nn.Module):
    def __init__(self, in_features, num_classes):
        super(DetectionHead, self).__init__()
        self.fc1 = nn.Linear(in_features, 1024)
        self.fc2 = nn.Linear(1024, 512)
        self.cls_score = nn.Linear(512, num_classes)
        self.bbox_pred = nn.Linear(512, num_classes * 4)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        class_logits = self.cls_score(x)
        bbox_regression = self.bbox_pred(x)
        return class_logits, bbox_regression

# Criando o Modelo Completo
class FasterRCNN(nn.Module):
    def __init__(self, num_classes):
        super(FasterRCNN, self).__init__()
        self.backbone = CustomBackbone()
        self.rpn = RPN(256)
        self.roi_pooling = ROIPooling(output_size=(7, 7))
        self.head = DetectionHead(in_features=7 * 7 * 256, num_classes=num_classes)

    def forward(self, images):
        features = self.backbone(images)
        rpn_logits, rpn_bbox = self.rpn(features)
        proposals = torch.rand((5, 4))  # Placeholder
        roi_features = self.roi_pooling(features, proposals)
        roi_features = roi_features.view(roi_features.size(0), -1)
        class_logits, bbox_regression = self.head(roi_features)
        return class_logits, bbox_regression

# Criando o modelo
model = FasterRCNN(num_classes=nc + 1)
model.to(device)
print("✅ Modelo Faster R-CNN criado do zero!")

# Criar Dataset
class ObjectDetectionDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        self.image_filenames = sorted([f for f in os.listdir(image_dir) if f.endswith(('.png', '.jpg', '.jpeg'))])
        self.label_dir = image_dir.replace("images", "labels")

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.image_filenames[idx])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image

# Criar DataLoaders
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = ObjectDetectionDataset(train_dir, transform=transform)
val_dataset = ObjectDetectionDataset(val_dir, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

# Configurar otimizador e treinar modelo
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
num_epochs = 5
loss_history = []

print("🚀 Iniciando treinamento...")
model.train()

for epoch in range(num_epochs):
    total_loss = 0

    for batch_idx, images in enumerate(train_loader):
        images = images.to(device)
        optimizer.zero_grad()
        class_logits, bbox_regression = model(images)
        loss = torch.mean(class_logits) + torch.mean(bbox_regression)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        if batch_idx % 10 == 0:
            print(f"🟢 Batch {batch_idx + 1}/{len(train_loader)} - Loss: {loss.item():.4f}")

    loss_history.append(total_loss)
    print(f"✅ Época {epoch + 1}/{num_epochs} finalizada. Loss Total: {total_loss:.4f}")

# Salvar modelo treinado
torch.save(model.state_dict(), "faster_rcnn_scract.pth")
print("💾 Modelo salvo com sucesso!")

# 📊 Box Plot da Loss ao longo das épocas
plt.figure(figsize=(8, 5))
plt.boxplot(loss_history, vert=True, patch_artist=True)
plt.xlabel('Épocas')
plt.ylabel('Loss Total')
plt.title('Distribuição da Loss ao Longo do Treinamento')
plt.grid()
plt.show()

# 📌 Testando uma imagem
def test_image(image_path):
    model.eval()
    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        class_logits, bbox_regression = model(image_tensor)

    plt.figure(figsize=(10, 6))
    plt.imshow(image)
    plt.title(f"Classe prevista: {torch.argmax(class_logits)}")
    plt.axis("off")
    plt.show()

# Teste com algumas imagens
print("🖼️ Testando algumas imagens...")
test_images = sorted([f for f in os.listdir(val_dir) if f.endswith(('.png', '.jpg', '.jpeg'))])[:3]

if test_images:
    for img_name in test_images:
        print(f"🟡 Testando imagem: {img_name}")
        test_image(os.path.join(val_dir, img_name))
else:
    print("⚠️ Nenhuma imagem válida encontrada para teste!")

print("✅ Teste finalizado!")
